# 01 · TorNet Veri Setini Tanıma

**Amaç:** Model eğitmeden önce veriyi anlamak.

1. `catalog.csv` dosyasını okuyup (203.133 örneğin tamamı, 38 MB) her yıldaki örnek ve tornado sayısını çıkarmak. **Radar verisini indirmeden.**
2. Veri setinin hazır eğitim/test ayrımında aynı fırtınanın iki tarafta da olup olmadığını kontrol etmek.
3. Gerçek radar görüntülerine bakmak: bir tornado, tornado çıkmamış bir uyarı ve rastgele bir fırtına.
4. Ön işlemenin önemli bilgiyi koruduğunu kontrol etmek.

Colab'da (CPU yeterli) ya da kendi bilgisayarında çalışır. Sadece birkaç yüz MB indirilir.

In [ ]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    if not REPO.exists():
        token = userdata.get("GITHUB_TOKEN")          # Colab Secrets (soldaki anahtar simgesi)
        !git clone -q https://{token}@github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q h5netcdf h5py
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))
print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import paths, tornet, viz

cfg = paths.load_config()
DATA = paths.data_dir()
print("veri klasörü:", DATA)
print("ayarlar:", cfg)

## 1. Katalog

Her satır bir örnek. `category` sütununun üç değeri var:

| category | anlamı | bizim etiketimiz |
|---|---|---|
| `TOR` | doğrulanmış tornado | **1** |
| `WRN` | tornado uyarısı verilmiş ama tornado çıkmamış | 0 |
| `NUL` | rastgele seçilmiş fırtına, tornado yok | 0 |

`WRN` örnekleri **zor negatifler**: meteorologların uyarı verecek kadar tehlikeli bulduğu ama tornado çıkmayan fırtınalar.

In [ ]:
cat = tornet.get_catalog(DATA)
print(f"{len(cat):,} örnek")
cat.head(3)

Yıl yıl dağılım. `tornado_%` sınıf dengesizliğini, `TOR_per_GB` indirilen her GB başına kaç tornado örneği düştüğünü gösterir.

In [ ]:
per_year = pd.crosstab(cat["year"], cat["category"])
per_year["total"] = per_year.sum(axis=1)
per_year["tornado_%"] = (100 * per_year["TOR"] / per_year["total"]).round(1)
sizes_gb = {2013: 3.2, 2014: 15.1, 2015: 17.4, 2016: 16.0, 2017: 15.1,
            2018: 12.5, 2019: 18.2, 2020: 17.0, 2021: 18.0, 2022: 19.0}   # Zenodo'daki arşiv boyutu
per_year["archive_GB"] = pd.Series(sizes_gb)
per_year["TOR_per_GB"] = (per_year["TOR"] / per_year["archive_GB"]).round(0)
per_year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5), layout="constrained")
per_year["TOR"].plot.bar(ax=ax, color=["C1" if y in cfg["years"] else "C7" for y in per_year.index])
ax.set_xlabel("yıl")
ax.set_ylabel("tornado örneği")
ax.set_title("Yıllara göre tornado örneği (turuncu = projede kullanılan yıllar)")
plt.show()

### Projede kullanılan yıllar

2013 küçük, çünkü veri Ağustos 2013'ün sonunda başlıyor. İndirilen GB başına en çok tornado örneği 2019 ve 2017'de. 2015 ise veri miktarını artırıyor.

In [ ]:
sel = cat[cat["year"].isin(cfg["years"])]
summary = pd.crosstab(sel["type"], sel["category"], margins=True)
print("Yıllar:", cfg["years"], f"| {len(sel):,} örnek | {sel['label'].sum():,} tornado (%{100*sel['label'].mean():.1f})")
summary

### Tornado şiddeti (EF ölçeği)

EF0 en zayıf, EF4 en güçlülerden. Modelimiz EF derecesini tahmin etmeyecek, ama sonra modelin güçlü tornadoları daha kolay bulup bulmadığına bakabiliriz.

In [ ]:
tor = cat[cat["category"] == "TOR"]
pd.crosstab(tor["year"], tor["ef_number"].astype(int), margins=True)

## 2. Eğitim/test ayrımı kontrolü

Veri seti hazır bir `type` sütunuyla geliyor (`train` / `test`). Aynı fırtına olayı (`episode_id`) iki tarafta da varsa
model daha önce gördüğü bir fırtınayla test edilmiş olur. Buna **veri sızıntısı** denir ve başarıyı olduğundan yüksek gösterir.
Bu fırtınalar eğitim kümesinden çıkarılacak.

In [ ]:
train_eps = set(cat.loc[cat["type"] == "train", "episode_id"])
test_eps = set(cat.loc[cat["type"] == "test", "episode_id"])
overlap = train_eps & test_eps
print(f"test fırtınası: {len(test_eps):,} | eğitimde de olan: {len(overlap)}")
print(f"bu fırtınalara ait eğitim örneği: {cat[(cat['type']=='train') & cat['episode_id'].isin(overlap)].shape[0]}")

## 3. Gerçek radar örneklerine bakış

2017 arşivinin sadece başını okuyup her kategoriden birkaç örnek bulunca duruyoruz (15 GB yerine birkaç yüz MB).

Bir örneğin içinde ne var (dosyalardan kontrol edildi):

* 6 radar değişkeni, her biri `(time=4, azimuth=120, range=240, sweep=2)` boyutunda
* 5 dakika arayla 4 zaman adımı; 2 tarama açısı (0,5° ve 0,9°)
* fırtınanın çevresinde 60° × 60 km'lik bir pencere
* ölçüm yapılamayan pikseller `NaN`

İki değişken kullanıyoruz:

* **DBZ (yansıtıcılık):** Havada ne kadar yağmur/dolu olduğunu gösterir. Tornadolu fırtınalarda sık sık **kanca** şekli görülür.
* **VEL (radyal hız):** Rüzgarın radara doğru (mavi) ya da radardan uzağa (kırmızı) esmesi. Tornado bir **hız çifti** olarak görünür:
  güçlü mavi ve güçlü kırmızının yan yana durması, yani dönme.

In [ ]:
samples = tornet.sample_from_tar(tornet.tar_url(2017), per_category={"TOR": 2, "WRN": 1, "NUL": 1})
for name in samples:
    print(name)

Bir örnek dosyanın içeriği:

In [ ]:
ds0 = next(iter(samples.values()))
ds0

In [ ]:
for name, ds in samples.items():
    viz.plot_sample(ds)
    plt.show()

Not: Zayıf tornadolar (EF0, EF1) radarda her zaman gözle belli olmaz. Görevi zorlaştıran şey de bu.

### Bir tornado örneğinin 4 zaman adımı

`frame_labels` tornadonun yerde olduğu kareleri işaretler. Örneğin etiketi **son** karenin etiketidir.
Önceki kareler ileride, RNN/LSTM haftalarında işe yarayabilir.

In [ ]:
tor_name = next(n for n in samples if "/TOR_" in n)
viz.plot_frames(samples[tor_name], "VEL")
plt.show()
viz.plot_frames(samples[tor_name], "DBZ")
plt.show()

## 4. Ağın göreceği girdi

Ön işleme adımları (`src/tornet.py`):

1. **DBZ ve VEL**, **en alçak tarama (0,5°)** ve **son zaman adımı** alınır → boyut `(2, 120, 240)`.
2. Değerler **sabit fiziksel aralıklarla** ölçeklenir (DBZ −20…60 dBZ, VEL −60…60 m/s). Verinin kendi istatistikleri kullanılmaz, böylece test kümesinden bilgi sızmaz.
3. `uint8` olarak saklanır (piksel başına 1 bayt, 4 kat daha küçük). `0` değeri boş pikseller için ayrılmıştır.
4. Yüklenirken `[-1, 1]` aralığına çevrilir ve boş pikseller fiziksel olarak anlamlı bir değerle doldurulur: DBZ → en düşük değer (yankı yok), VEL → 0 (hareket yok).

Ağ kutupsal ızgarayı doğrudan görür (satırlar = açı, sütunlar = mesafe). Bu yüzden şekiller yukarıdaki haritaya göre gerilmiş görünür.

In [ ]:
ds = samples[tor_name]
q = tornet.preprocess_sample(ds, cfg["variables"], cfg["sweep"], cfg["frame"])
x = tornet.from_uint8(q, cfg["variables"])
print("saklanan:", q.shape, q.dtype, "| ağın girdisi:", x.shape, x.dtype, f"[{x.min():.2f}, {x.max():.2f}]")
print("kanal başına boş piksel (%):", (100 * (q == 0).mean(axis=(1, 2))).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
for ax, v, img in zip(axes, cfg["variables"], x):
    m = ax.imshow(img, aspect="auto", cmap=viz.CMAPS[v], vmin=-1, vmax=1)
    ax.set_title(f"{v} (ağın girdisi)")
    ax.set_xlabel("mesafe hücresi (250 m)")
    ax.set_ylabel("açı hücresi (0,5°)")
    plt.colorbar(m, ax=ax)
plt.show()

## Özet

* TorNet: 203.133 örnek, 13.857 tornado (%6,8). Sınıflar çok dengesiz, bu yüzden tek başına doğruluk (accuracy) yanıltıcı olur.
* Kullanılan yıllar yukarıdaki tabloda. Test örnekleri her yılın resmi `test` kümesinden gelecek. Ortak fırtınalar eğitimden çıkarılacak.
* Sonraki adım: `02_preprocess.ipynb` seçilen yılları Colab'da indirir ve küçültülmüş dizileri Google Drive'a kaydeder.